In [0]:
from pyspark.sql.functions import *


FILE_PATH=("/Volumes/final_capstone_project/source/input/Vehicle Service Incremental.csv")

vehicle_service_increment=("final_capstone_project.bronze.vehicle_bronze_increment")

rejected_service_increment=("final_capstone_project.bronze.retected_bronze_increment")


In [0]:
vehicle_service_increment=(
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(FILE_PATH)
)

In [0]:
display(vehicle_service_increment)

In [0]:
vehicle_service_increment=(
    vehicle_service_increment
    .withColumn("source_file",lit("vehicle2"))
    .withColumn("batch_id",lit("batch2"))
    .withColumn("ingestion_timestamp",current_timestamp())
)


In [0]:
display(vehicle_service_increment)

In [0]:
vehicle_service_increment.printSchema()


In [0]:
from pyspark.sql import functions as F

vehicle_service_increment= (
    vehicle_service_increment
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("service_id").isNull() | (F.trim(F.col("service_id")) == ""),
            "Missing service ID"
        )
        .when(
            F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""),
            "Missing customer ID"
        )
        .when(
            F.col("vehicle_id").isNull() | (F.trim(F.col("vehicle_id")) == ""),
            "Missing vehicle ID"
        )
        .when(
            F.col("service_type").isNull() | (F.trim(F.col("service_type")) == ""),
            "Missing service type"
        )
        .when(
            F.col("service_city").isNull() | (F.trim(F.col("service_city")) == ""),
            "Missing service city"
        )
        .when(
            F.col("vehicle_type").isNull() |
            (~F.upper(F.trim(F.col("vehicle_type"))).isin("SUV", "EV", "SEDAN", "HATCHBACK")),
            "Invalid vehicle type"
        )
        .when(
            F.col("service_status").isNull() |
            (~F.upper(F.trim(F.col("service_status"))).isin("OPEN", "COMPLETED")),
            "Invalid service status"
        )
        .when(
            F.col("labour_charge").isNull() | (F.col("labour_charge") <= 0.0),
            "Invalid labour charge"
        )
        .when(
            F.col("parts_charge").isNull() | (F.col("parts_charge") <= 0.0),
            "Invalid parts charge"
        )
        .when(
            F.col("discount").isNull() | (F.col("discount") < 0.0),
            "Invalid discount amount"
        )
        .when(
            F.to_date(F.col("service_date"), "yyyy-MM-dd").isNull(),
            "Invalid service date"
        )
        .otherwise(F.lit(None))
    )
)

In [0]:
rejected_service_increment= vehicle_service_increment.filter(F.col("rejection_reason").isNotNull())

vehicle_service_increment= (
    vehicle_service_increment.filter(F.col("rejection_reason").isNull())
      .drop("rejection_reason")
)

In [0]:
display(vehicle_service_increment)


In [0]:
display(rejected_service_increment)

In [0]:
print("source_count:",vehicle_service_increment.count())

In [0]:
print("good count:",vehicle_service_increment.count())

In [0]:
print("bad count:",rejected_service_increment.count())

In [0]:
#Reconciliation
source_count = spark.read.option("header", True).csv(FILE_PATH).count()

good_count = vehicle_service_increment.count()

bad_count = rejected_service_increment.count()

print("Source :", source_count)
print("Good   :", good_count)
print("Bad    :", bad_count)

if source_count == good_count + bad_count:
    print("PASS: Reconciliation successful")
else:
    raise Exception(
        "FAILED: Source reconciliation failed"
    )


In [0]:
# Check count first to confirm 94 rows are present before deduplication
print("Count before dedup:", vehicle_service_increment.count())

# 1. Identify duplicate service_ids
duplicate_ids = (
    vehicle_service_increment
    .groupBy("service_id")
    .count()
    .filter(F.col("count") > 1)
    .select("service_id")
)

# 2. Show the duplicate rows separately (should display 4 rows: SRV0025 x2, SRV0050 x2)
duplicate_service = vehicle_service_increment.join(duplicate_ids, on="service_id", how="inner")
display(duplicate_service)

# 3. Drop duplicates AFTER displaying them
vehicle_service_increment = vehicle_service_increment.dropDuplicates(["service_id"])
print("Count after dedup:", vehicle_service_increment.count())

In [0]:
vehicle_service_increment.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("header","true")\
    .saveAsTable("final_capstone_project.bronze.vehicle_bronze_increment")

In [0]:
rejected_service_increment.write\
    .format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", "true")\
    .option("header","true")\
    .saveAsTable("final_capstone_project.bronze.rejected_services_increment")

In [0]:
display(vehicle_service_increment)

In [0]:
display(rejected_service_increment)